# SFT Training Launcher

Colab 런타임에서 `scripts/train_sft.py`를 실행하기 위한 얇은 런처 노트북입니다.

GitHub repo와 실험 설정입니다. Colab에서 실행하기 전에 이 셀의 값만 수정하세요.

In [9]:
GITHUB_REPO_URL = "https://github.com/c-peace/LEET_LLM_PEFT.git"
GITHUB_BRANCH = "main"
PROJECT_DIR = "/content/seonji-llm"

HF_USERNAME = "choipeace"
PROJECT_NAME = "LEET_seonji"
EXPERIMENT_VERSION = "aug-v1"
EXPERIMENT_TITLE = "augmented-train-fixed-test"
EXPERIMENT_HYPOTHESIS = """증강 학습셋으로 부족 유형을 보강하면 고정 test_set_v1 평가에서 선지 생성 품질이 개선되는지 확인한다."""
HF_REPO_PRIVATE = True

OUTPUT_ROOT = "/content/sft_outputs/runs"
BASE_CONFIG_PATH = "configs/train_config_v1.json"
TMP_CONFIG_PATH = "/content/train_config_no_drive.json"


GitHub에서 프로젝트 코드를 가져오고 작업 폴더로 이동합니다.

In [10]:
import os
import subprocess
from pathlib import Path

if "YOUR_GITHUB_ID" in GITHUB_REPO_URL:
    raise ValueError("GITHUB_REPO_URL을 본인 GitHub repo 주소로 바꾼 뒤 실행하세요.")

project_dir = Path(PROJECT_DIR)
if project_dir.exists():
    subprocess.run(["git", "-C", str(project_dir), "pull"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, str(project_dir)], check=True)

os.chdir(project_dir)
print(f"cwd={Path.cwd()}")
required_paths = [
    "scripts/train_sft.py",
    "scripts/upload_hf_adapter.py",
    "sft_train_aug_v1.json",
    "test_set_v1.json",
    "configs/train_config_v1.json",
]
for required_path in required_paths:
    path = Path(required_path)
    if not path.exists():
        raise FileNotFoundError(f"required file not found: {path}")
    print(f"found: {path}")


cwd=/content/seonji-llm
found: scripts/train_sft.py
found: scripts/upload_hf_adapter.py
found: sft_dataset.json
found: configs/train_config_v1.json


런타임 GPU를 확인합니다.

In [11]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


필요 패키지를 설치합니다.

In [12]:
!pip install -U transformers datasets peft accelerate bitsandbytes huggingface_hub

Hugging Face Hub에 로그인합니다. 토큰은 노트북 입력창에 직접 붙여넣고, 코드나 채팅에 남기지 마세요.

In [13]:
from huggingface_hub import notebook_login, whoami

notebook_login()

user = whoami()
print(f"Logged in to Hugging Face as: {user['name']}")

LocalTokenNotFoundError: Token is required to call the /whoami-v2 endpoint, but no token found. You must provide a token or be logged in to Hugging Face with `hf auth login` or `huggingface_hub.login`. See https://huggingface.co/settings/tokens.

Drive 용량을 쓰지 않도록 학습 결과 저장 위치를 Colab 임시 디스크로 바꿉니다.

In [ ]:
import json
from pathlib import Path

config_path = Path(BASE_CONFIG_PATH)
config = json.loads(config_path.read_text(encoding="utf-8"))
config["outputs"]["root_dir"] = OUTPUT_ROOT
config["training"]["save_strategy"] = "no"

if config["dataset"]["path"] != "sft_train_aug_v1.json":
    raise ValueError(f"unexpected train dataset: {config['dataset']['path']}")
if config["dataset"].get("test_path") != "test_set_v1.json":
    raise ValueError(f"unexpected test dataset: {config['dataset'].get('test_path')}")

tmp_config_path = Path(TMP_CONFIG_PATH)
tmp_config_path.write_text(
    json.dumps(config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"temporary config: {tmp_config_path}")
print(f"train dataset: {config['dataset']['path']}")
print(f"final test dataset: {config['dataset']['test_path']}")
print(f"output root: {config['outputs']['root_dir']}")
print(f"save_strategy: {config['training']['save_strategy']}")
print(f"experiment: {PROJECT_NAME}-{EXPERIMENT_VERSION} / {EXPERIMENT_TITLE}")


학습이 끝난 최신 run을 Hugging Face Hub에 업로드하는 helper 함수입니다. 각 모델 학습 셀 끝에서 자동 호출됩니다.

In [ ]:
import subprocess

def upload_latest_run():
    if HF_USERNAME == "YOUR_HF_USERNAME":
        raise ValueError("HF_USERNAME을 본인 Hugging Face 아이디로 바꾼 뒤 실행하세요.")

    subprocess.run(
        [
            "python",
            "scripts/upload_hf_adapter.py",
            "--hf_username",
            HF_USERNAME,
            "--project_name",
            PROJECT_NAME,
            "--experiment_version",
            EXPERIMENT_VERSION,
            "--experiment_title",
            EXPERIMENT_TITLE,
            "--experiment_hypothesis",
            EXPERIMENT_HYPOTHESIS,
            "--runs_root",
            OUTPUT_ROOT,
            "--private",
            str(HF_REPO_PRIVATE).lower(),
        ],
        check=True,
    )

Qwen augmented-train fixed-test 학습 실행


In [ ]:
!python scripts/train_sft.py \
  --config {TMP_CONFIG_PATH} \
  --model_name Qwen/Qwen3.5-4B \
  --copy_config

upload_latest_run()

# smoke test
# !python scripts/train_sft.py \
#   --config {TMP_CONFIG_PATH} \
#   --model_name Qwen/Qwen3.5-4B \
#   --run_id smoke_qwen \
#   --max_steps 10 \
#   --eval_sample_size 5 \
#   --copy_config
# upload_latest_run()


Gemma augmented-train fixed-test 학습 실행


In [ ]:
!python scripts/train_sft.py \
  --config {TMP_CONFIG_PATH} \
  --model_name google/gemma-4-E4B-it \
  --copy_config

upload_latest_run()

EXAONE augmented-train fixed-test 학습 실행


In [ ]:
!python scripts/train_sft.py \
  --config {TMP_CONFIG_PATH} \
  --model_name LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct \
  --copy_config

upload_latest_run()

최근 결과 폴더와 평가 요약을 확인합니다.

In [ ]:
import json
from pathlib import Path

runs = sorted(Path(OUTPUT_ROOT).glob("*"), key=lambda p: p.stat().st_mtime, reverse=True)
for run in runs[:5]:
    print(run)

if runs:
    latest_run = runs[0]
    summary_path = latest_run / "eval_summary.json"
    results_path = latest_run / "eval_results.json"
    print()
    print(f"latest_run={latest_run}")
    if summary_path.exists():
        print(json.dumps(json.loads(summary_path.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))
    if results_path.exists():
        results = json.loads(results_path.read_text(encoding="utf-8"))
        print(f"train_dataset={results.get('dataset_path')}")
        print(f"eval_dataset={results.get('eval_dataset_path')}")


다른 컴퓨터나 다른 Colab에서 adapter를 불러오는 예시입니다.

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model = "Qwen/Qwen3.5-4B"
adapter_repo = "choipeace/leet-seonji-qwen35-4b-v1"

tokenizer = AutoTokenizer.from_pretrained(adapter_repo)
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    device_map="auto",
)
model = PeftModel.from_pretrained(model, adapter_repo)

In [ ]:
from google.colab import runtime

runtime.unassign()
